# dYdX collector catalog -> pandas

Reads the collector's `ParquetDataCatalog` output directly through Nautilus's own catalog API, which already decodes the fixed-point `Price`/`Quantity` blobs and derives `instrument_id` -- no manual byte-decoding needed (unlike querying the raw Parquet files with a generic tool). `to_dict()` is a staticmethod on each `Data`/`Instrument` class, so it's called as `TradeTick.to_dict(t)`, not `t.to_dict()`.

In [ ]:
import pandas as pd

from nautilus_trader.model.data import TradeTick
from nautilus_trader.model.instruments import CryptoPerpetual
from nautilus_trader.persistence.catalog import ParquetDataCatalog


catalog = ParquetDataCatalog("../catalog")  # relative to this notebook's directory
instrument_ids = [i.id.value for i in catalog.instruments()]
instrument_ids

In [ ]:
instruments_df = pd.DataFrame([CryptoPerpetual.to_dict(i) for i in catalog.instruments()])
instruments_df

In [ ]:
instrument_id = instrument_ids[0]
trades = catalog.trade_ticks(instrument_ids=[instrument_id])
trades_df = pd.DataFrame([TradeTick.to_dict(t) for t in trades])

# price/size come back as exact strings (avoids float precision loss in storage);
# cast to float here for plotting/arithmetic.
trades_df["price"] = trades_df["price"].astype(float)
trades_df["size"] = trades_df["size"].astype(float)
trades_df["ts_event"] = pd.to_datetime(trades_df["ts_event"], unit="ns")
trades_df.set_index("ts_event").sort_index()

In [ ]:
trades_df.set_index("ts_event").sort_index()["price"].plot(title=instrument_id)